<a href="https://colab.research.google.com/github/AnanyaAsthana/Hadoop-CUDA-Lab/blob/main/hadoopCudaLab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi


Wed Jan 21 09:28:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile vector_add.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <sys/time.h>

// CUDA Kernel
__global__ void vectorAdd(float *A, float *B, float *C, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n)
        C[idx] = A[idx] + B[idx];
}

// CPU function
void vectorAddCPU(float *A, float *B, float *C, int n) {
    for (int i = 0; i < n; i++)
        C[i] = A[i] + B[i];
}

double getTime() {
    struct timeval tv;
    gettimeofday(&tv, NULL);
    return tv.tv_sec + tv.tv_usec * 1e-6;
}

int main() {
    int n;
    printf("Enter number of elements: ");
    scanf("%d", &n);

    size_t size = n * sizeof(float);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C_cpu = (float*)malloc(size);
    float *h_C_gpu = (float*)malloc(size);

    for (int i = 0; i < n; i++) {
        h_A[i] = i;
        h_B[i] = i * 2;
    }

    double cpu_start = getTime();
    vectorAddCPU(h_A, h_B, h_C_cpu, n);
    double cpu_end = getTime();

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    double gpu_start = getTime();
    vectorAdd<<<blocks, threads>>>(d_A, d_B, d_C, n);
    cudaDeviceSynchronize();
    double gpu_end = getTime();

    cudaMemcpy(h_C_gpu, d_C, size, cudaMemcpyDeviceToHost);

    printf("\nCPU Time: %f seconds", cpu_end - cpu_start);
    printf("\nGPU Time: %f seconds\n", gpu_end - gpu_start);

    free(h_A); free(h_B); free(h_C_cpu); free(h_C_gpu);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);

    return 0;
}


Writing vector_add.cu


In [ ]:
!nvcc vector_add.cu -o vector_add


In [ ]:
!./vector_add


Enter number of elements: 100

CPU Time: 0.000000 seconds
GPU Time: 0.043315 seconds


In [ ]:
!./vector_add


Enter number of elements: 1000

CPU Time: 0.000005 seconds
GPU Time: 0.011202 seconds


In [ ]:
!./vector_add



Enter number of elements: 5000

CPU Time: 0.000021 seconds
GPU Time: 0.007368 seconds


In [ ]:
!./vector_add


Enter number of elements: 50000

CPU Time: 0.000233 seconds
GPU Time: 0.007778 seconds


In [ ]:
!./vector_add


Enter number of elements: 70000

CPU Time: 0.000317 seconds
GPU Time: 0.007621 seconds


In [ ]:
!./vector_add


Enter number of elements: 100000

CPU Time: 0.000552 seconds
GPU Time: 0.007110 seconds


In [ ]:
!./vector_add


Enter number of elements: 200000

CPU Time: 0.001153 seconds
GPU Time: 0.007302 seconds


In [ ]:
!./vector_add


Enter number of elements: 500000

CPU Time: 0.002280 seconds
GPU Time: 0.007411 seconds


In [ ]:
!./vector_add


Enter number of elements: 1000000

CPU Time: 0.004765 seconds
GPU Time: 0.007308 seconds


In [ ]:
!./vector_add


Enter number of elements: 2000000

CPU Time: 0.009001 seconds
GPU Time: 0.007094 seconds


In [ ]:
!./vector_add


Enter number of elements: 1500000

CPU Time: 0.007104 seconds
GPU Time: 0.007446 seconds


In [ ]:
!./vector_add


Enter number of elements: 1750000

CPU Time: 0.007859 seconds
GPU Time: 0.007395 seconds


In [ ]:
!./vector_add


Enter number of elements: 1700000

CPU Time: 0.007885 seconds
GPU Time: 0.007956 seconds


In [ ]:
!./vector_add


Enter number of elements: 1725000

CPU Time: 0.007756 seconds
GPU Time: 0.007444 seconds


In [ ]:
!./vector_add


Enter number of elements: 1720000

CPU Time: 0.008194 seconds
GPU Time: 0.007489 seconds


In [ ]:
!./vector_add


Enter number of elements: 3000000

CPU Time: 0.013795 seconds
GPU Time: 0.007389 seconds


In [ ]:
!./vector_add


Enter number of elements: 1700000

CPU Time: 0.007866 seconds
GPU Time: 0.007742 seconds


In [ ]:
%%writefile vector_sum_max_min.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <sys/time.h>
#include <limits.h>

__global__ void sumKernel(int *arr, int *result, int n) {
    __shared__ int temp[256];
    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + tid;

    temp[tid] = (idx < n) ? arr[idx] : 0;
    __syncthreads();

    for (int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (tid < s)
            temp[tid] += temp[tid + s];
        __syncthreads();
    }

    if (tid == 0)
        atomicAdd(result, temp[0]);
}

__global__ void maxMinKernel(int *arr, int *maxVal, int *minVal, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        atomicMax(maxVal, arr[idx]);
        atomicMin(minVal, arr[idx]);
    }
}

void cpuSumMaxMin(int *arr, int n, int *sum, int *max, int *min) {
    *sum = 0;
    *max = INT_MIN;
    *min = INT_MAX;

    for (int i = 0; i < n; i++) {
        *sum += arr[i];
        if (arr[i] > *max) *max = arr[i];
        if (arr[i] < *min) *min = arr[i];
    }
}

double getTime() {
    struct timeval tv;
    gettimeofday(&tv, NULL);
    return tv.tv_sec + tv.tv_usec * 1e-6;
}

int main() {
    int n;
    printf("Enter number of elements: ");
    scanf("%d", &n);

    size_t size = n * sizeof(int);
    int *h_arr = (int*)malloc(size);

    for (int i = 0; i < n; i++)
        h_arr[i] = i + 1;

    int cpuSum, cpuMax, cpuMin;
    double cpu_start = getTime();
    cpuSumMaxMin(h_arr, n, &cpuSum, &cpuMax, &cpuMin);
    double cpu_end = getTime();

    int *d_arr, *d_sum, *d_max, *d_min;
    cudaMalloc(&d_arr, size);
    cudaMalloc(&d_sum, sizeof(int));
    cudaMalloc(&d_max, sizeof(int));
    cudaMalloc(&d_min, sizeof(int));

    cudaMemcpy(d_arr, h_arr, size, cudaMemcpyHostToDevice);
    cudaMemset(d_sum, 0, sizeof(int));

    int initMax = INT_MIN, initMin = INT_MAX;
    cudaMemcpy(d_max, &initMax, sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_min, &initMin, sizeof(int), cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    double gpu_start = getTime();
    sumKernel<<<blocks, threads>>>(d_arr, d_sum, n);
    maxMinKernel<<<blocks, threads>>>(d_arr, d_max, d_min, n);
    cudaDeviceSynchronize();
    double gpu_end = getTime();

    int gpuSum, gpuMax, gpuMin;
    cudaMemcpy(&gpuSum, d_sum, sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(&gpuMax, d_max, sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(&gpuMin, d_min, sizeof(int), cudaMemcpyDeviceToHost);

    printf("\nCPU Results:");
    printf("\nSum = %d, Max = %d, Min = %d", cpuSum, cpuMax, cpuMin);
    printf("\nCPU Time: %f seconds\n", cpu_end - cpu_start);

    printf("\nGPU Results:");
    printf("\nSum = %d, Max = %d, Min = %d", gpuSum, gpuMax, gpuMin);
    printf("\nGPU Time: %f seconds\n", gpu_end - gpu_start);

    free(h_arr);
    cudaFree(d_arr);
    cudaFree(d_sum);
    cudaFree(d_max);
    cudaFree(d_min);

    return 0;
}


Writing vector_sum_max_min.cu


In [ ]:
!nvcc vector_sum_max_min.cu -o vector_sum_max_min


In [ ]:
!./vector_sum_max_min


Enter number of elements: 1000

CPU Results:
Sum = 500500, Max = 1000, Min = 1
CPU Time: 0.000005 seconds

GPU Results:
Sum = 0, Max = -2147483648, Min = 2147483647
GPU Time: 0.007580 seconds


In [ ]:
!./vector_sum_max_min


Enter number of elements: 10000

CPU Results:
Sum = 50005000, Max = 10000, Min = 1
CPU Time: 0.000044 seconds

GPU Results:
Sum = 0, Max = -2147483648, Min = 2147483647
GPU Time: 0.007728 seconds


In [ ]:
!./vector_sum_max_min


Enter number of elements: 100000

CPU Results:
Sum = 705082704, Max = 100000, Min = 1
CPU Time: 0.000469 seconds

GPU Results:
Sum = 0, Max = -2147483648, Min = 2147483647
GPU Time: 0.007456 seconds


In [ ]:
!./vector_sum_max_min


Enter number of elements: 1000000

CPU Results:
Sum = 1784293664, Max = 1000000, Min = 1
CPU Time: 0.004583 seconds

GPU Results:
Sum = 0, Max = -2147483648, Min = 2147483647
GPU Time: 0.007349 seconds


In [ ]:
!./vector_sum_max_min


Enter number of elements: 1500000

CPU Results:
Sum = -280681552, Max = 1500000, Min = 1
CPU Time: 0.006733 seconds

GPU Results:
Sum = 0, Max = -2147483648, Min = 2147483647
GPU Time: 0.008417 seconds


In [ ]:
!./vector_sum_max_min


Enter number of elements: 1700000

CPU Results:
Sum = 1891838544, Max = 1700000, Min = 1
CPU Time: 0.007741 seconds

GPU Results:
Sum = 0, Max = -2147483648, Min = 2147483647
GPU Time: 0.007512 seconds
